# Custom Datasets

PyTorch has domain libraries for multiple tipes of data
* `torchvision`: For image datasets
* `torchtext`: for text datasets
* `torchaudio`: for audio datasets
* `torchrec`: for recomendations systems
* `torchdata`: For other kinds of data

### Food 101 Data Set

101 food classes and 101.000 images
    - For each class, 250 are test data (manually reviewed) and 750 training data (un-cleanned, containing noise and wrong labels)
    - All images have 512 pixels as maximum side length

`learnpytorch.io` has a subset of this dataset, with only 3 food categories and 300 images, in the same test/training proportion

In [ ]:
import requests
import zipfile
from pathlib import Path

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    # Download pizza, steak, sushi data
    with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza, steak, sushi data...")
        f.write(request.content)

    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping pizza, steak, sushi data...")
        zip_ref.extractall(image_path)

In [ ]:
import os
def walk_through_dir(dir_path):
  """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str or pathlib.Path): target directory

  Returns:
    A print out of:
      number of subdiretories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

print(walk_through_dir(image_path))

## Visualizing the Custom Dataset

1. Visualize the downloaded dataset path
2. For examplification, pick a random file using `random.choce()`
3. Get the class name from the file using `pathlib.Path.parent.stem`
    * assuming the fil categorie is the imediate parent folder
4. Open the file (for images, using Python's `PIL`)
5. Display the file and it's metadata
6. Visualize it with a suited tool (like matplotlib)

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"
print("Train Dir:", train_dir, "Test Dir:", test_dir)

import random
from PIL import Image
import torch
RND_SEED = 0
# random.seed(RND_SEED)
torch.manual_seed(RND_SEED)
torch.cuda.manual_seed(RND_SEED)

img_pth_list = list(image_path.glob("*/*/*.jpg"))
print("Example Paths", img_pth_list[:3])

rnd_img_pth = random.choice(img_pth_list)
print("Random Path", rnd_img_pth)

rnd_img_class = rnd_img_pth.parent.stem
print("Random Class", rnd_img_class)

rnd_img = Image.open(rnd_img_pth)
print("Image Dimentions", rnd_img.height, rnd_img.width)
rnd_img

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rnd_img_as_array = np.asarray(rnd_img)

plt.figure(figsize=(10, 7))
plt.imshow(rnd_img_as_array)
plt.title(f"Class: {rnd_img_class} | Array Shape {rnd_img_as_array.shape}", c="w", backgroundcolor="black") # 3 color channels, and the PIL default is color channels last
plt.axis(False);

# Transform the Custom Data to Pytorch

1. First turn into tensors
2. Then the tensors into `torch.utils.data.Dataset`, then into `torch.utils.data.DataLoader`
    * For images, it's possible to use the `transforms` module from torchvision to go from image to tensor (it converts HxWxC to CxHxW)

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets, transforms

HEIGHT = 64
WIDTH = 64

data_transformer = transforms.Compose([
    # Resize to 64x64, to reduce memory foot print
    transforms.Resize(size=(HEIGHT, WIDTH)),
    # Flips the images
    transforms.RandomHorizontalFlip(
        0.5 # 50% chance
    ),
    # Converts a PIL Image (C x H x W), range [0, 255]
    # Into a (C x H x W) torch.Tensor in range [0,1]
    transforms.ToTensor(),
])

data_transformer(rnd_img)

## Compare the transformed files with the original ones

In [ ]:
def plot_transformed_images(image_paths: list[Path], transform: transforms.Compose, n: int=3, seed:int =None):
    """Plots a series of random images from image_paths.

    Will open n image paths from image_paths, transform them
    with transform and plot them side by side.

    Args:
        image_paths (list): List of target image paths.
        transform (PyTorch Transforms): Transforms to apply to images.
        n (int, optional): Number of images to plot. Defaults to 3.
        seed (int, optional): Random seed for the random generator. Defaults to None for varying random values.
    """
    if seed:
        random.seed(seed)
    random_image_paths = random.sample(image_paths, k=n)
    for image_path in random_image_paths:
        with Image.open(image_path) as f:
            fig, ax = plt.subplots(1, 2)
            ax[0].imshow(f)
            ax[0].set_title(f"Original \nSize: {f.size}")
            ax[0].axis("off")

            # Transform and plot image
            # Note: permute() will change shape of image to suit matplotlib
            # (PyTorch default is [C, H, W] but Matplotlib is [H, W, C])
            transformed_image = transform(f).permute(1, 2, 0)
            ax[1].imshow(transformed_image)
            ax[1].set_title(f"Transformed \nSize: {transformed_image.shape}")
            ax[1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

plot_transformed_images(
    img_pth_list,
    transform=data_transformer,
    n=3, seed=RND_SEED
)

## Load the Custom Dataset


### Get a Dataset out of the Folder

For images, its possible to use `torchvision.datasets.ImageFolder` to get a dataset out of a folder directory with images as files

In [ ]:
from torchvision.datasets import ImageFolder

train_data = ImageFolder(
    root=train_dir,
    transform=data_transformer, # transform for the input
    target_transform=None, # transformer for the output
)
test_data = ImageFolder(
    root=test_dir,
    transform=data_transformer, # transform for the input
    target_transform=None, # transformer for the output
)

print(len(train_data), "train /", len(test_data), "test")
class_names = train_data.classes
print("Categories:", class_names)

img, label = train_data[0]

print("Image (CxHxW)", img.shape, "Label", label)
img_permuted = img.permute(1, 2, 0)
print("Image Permuted (HxWxC):", img_permuted.shape)

plt.figure(figsize=(10, 7))
plt.imshow(img_permuted)
plt.axis("off")
plt.title(class_names[label])

### Get a Dataloader out of a Dataset
A dataloader allow to customize batch size and iterate through the dataset

In [ ]:
import os
import torch
from torch.utils.data import DataLoader

BATCH_SIZE = 32
tmp = os.cpu_count()
NUM_WORKERS = tmp if tmp else 0

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)
test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)

print(len(train_dataloader), "train batches /", len(test_dataloader), "test batches")
print("Batches of Size:", train_dataloader.batch_size)

img_batch, label_batch = next(iter(train_dataloader))

print("Batch Shape:", f"{img_batch.shape} x {label_batch.shape}", "-> [batch-size, colors, height, width] x batch_size")

img = img_batch[0]
label = label_batch[0]

print("Image (CxHxW)", img.shape, "Label", label)
img_permuted = img.permute(1, 2, 0)
print("Image Permuted (HxWxC):", img_permuted.shape)

plt.figure(figsize=(10, 7))
plt.imshow(img_permuted)
plt.axis("off")
plt.title(class_names[label])

## Creating a Custom Dataloading Function

Not all datatypes may have a specialized function to load and prepare them

1. Capable of Loading data of a type from file directories
2. Getting Class Names from the Dataset
    * using `os.scandir` to traverse the directory
    * Raise errors if the class name isn't found
    * Return the classes names ass list and dictionaries

In [ ]:
import os
from pathlib import Path
from typing import Tuple, Dict, List

def find_classes(Dir: str|Path) -> Tuple[List[str], Dict[str, int]]:
    """
    Finds the class folder names in a target directory.

    Assumes target directory is in standard image classification format.

    Args:
        directory (str): target directory to load classnames from.

    Returns:
        Tuple[List[str], Dict[str, int]]: (list_of_class_names, dict(class_name: idx...))

    Example:
        find_classes("food_images/train")
        >>> (["class_1", "class_2"], {"class_1": 0, ...})
    """
    # 1. Get the class names by scanning the target directory
    classes = sorted(entry.name for entry in os.scandir(Dir) if entry.is_dir())

    # 2. Raise an error if class names not found
    if not classes:
        raise FileNotFoundError(f"Couldn't find any classes in {Dir}.")

    # 3. Create a dictionary of index labels (computers prefer numerical rather than string labels)
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
    return classes, class_to_idx

print(find_classes(train_dir))


### Custom Dataset Class

1. Will subclass `torch.utils.data.Dataset`

In [ ]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset

from PIL import Image
from pathlib import Path

class CustomDataSet(Dataset):
    """
    - Subclass torch.utils.data.Dataset.
    - Initialize our subclass with a targ_dir parameter and transform parameter.
    - Create several attributes for paths, transform, classes and class_to_idx.
    - Create a function to load data from file and return them.
    - Overwrite the __len__ method of torch.utils.data.Dataset to return the number of samples in the Dataset, allowing calling len(Dataset).
    - Overwrite the __getitem__ method of torch.utils.data.Dataset to return a single sample from the Dataset.
    """
    def __init__(self,
                target_dir: str|Path,
                data_ext="jpg",
                dtype=Image,
                transformer:transforms.Compose=None  # type: ignore
        ):
        super().__init__()
        # Get all image paths
        self.paths = list(Path(target_dir).glob(f"*/*.{data_ext}"))
        # Setup transforms
        if transformer is None:
            transformer = transforms.Compose([(transforms.ToTensor())])
        self.transform = transformer
        # Create classes and class_to_idx attributes
        self.classes, self.class_to_idx = find_classes(target_dir)

        if dtype == Image:
            self.open_method = Image.open
        else:
            raise TypeError("Data Type not supported")


    def load_data(self, index: int) -> torch.Tensor:
        data_path = self.paths[index]
        data = self.open_method(data_path)
        return self.transform(data) # type: ignore

    # Overwrite the __len__() method
    def __len__(self) -> int:
        "Returns the total number of samples."
        return len(self.paths)

    # Overwrite the __getitem__() method (required for subclasses of torch.utils.data.Dataset)
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        "Returns one sample of data, data and label (X, y)."
        data = self.load_data(index)
        class_name  = self.paths[index].parent.name
        class_idx = self.class_to_idx[class_name]

        # return data, label (X, y)
        return data, class_idx

In [ ]:
train_transf = transforms.Compose([
    # Resize to HEIGHTxWIDTH, to reduce memory foot print
    transforms.Resize(size=(HEIGHT, WIDTH)),
    # Flips the images
    transforms.RandomHorizontalFlip(
        0.5 # 50% chance
    ),
    # Converts a PIL Image (C x H x W), range [0, 255]
    # Into a (C x H x W) torch.Tensor in range [0,1]
    transforms.ToTensor(),
])
test_transf = transforms.Compose([
    transforms.Resize(size=(HEIGHT, WIDTH)),
    # Converts a PIL Image (C x H x W), range [0, 255]
    # Into a (C x H x W) torch.Tensor in range [0,1]
    transforms.ToTensor(),
])

train_data_custom = CustomDataSet(
    target_dir=train_dir,
    data_ext="jpg",
    dtype=Image,
    transformer=train_transf,
)
test_data_custom = CustomDataSet(
    target_dir=test_dir,
    data_ext="jpg",
    dtype=Image,
    transformer=test_transf,
)
print(len(train_data_custom), len(test_data_custom))
print(train_data_custom.classes, test_data_custom.classes)
print(train_data_custom.class_to_idx, test_data_custom.class_to_idx)
print(train_data_custom[0][0].shape, test_data_custom[0][0].shape)

import random

# 1. Take in a Dataset as well as a list of class names
def display_random_images(
        dataset: torch.utils.data.Dataset,
        classes: List[str] = [],
        n: int = 10,
        display_shape: bool = True,
        seed: int = None # type: ignore
    ):

    # 2. Adjust display if n too high
    if n > 10:
        n = 10
        display_shape = False
        print(f"For display purposes, n shouldn't be larger than 10, setting to 10 and removing shape display.")

    # 3. Set random seed
    if seed:
        random.seed(seed)

    # 4. Get random sample indexes
    random_samples_idx = random.sample(range(len(dataset)), k=n)

    # 5. Setup plot
    plt.figure(figsize=(16, 8))

    # 6. Loop through samples and display random samples
    for i, targ_sample in enumerate(random_samples_idx):
        targ_image, targ_label = dataset[targ_sample]#, dataset[targ_sample]

        # 7. Adjust image tensor shape for plotting: [color_channels, height, width] -> [color_channels, height, width]
        targ_image_adjust = targ_image.permute(1, 2, 0)

        # Plot adjusted samples
        plt.subplot(1, n, i+1)
        plt.imshow(targ_image_adjust)
        plt.axis("off")
        if classes:
            title = f"class: {classes[targ_label]}"
            if display_shape:
                title = title + f"\nshape: {targ_image_adjust.shape}"
        plt.title(title)

display_random_images(
    dataset=train_data_custom,
    classes=class_names,
    display_shape=False
)

### Create a DataLoader from Custom Dataset

In [ ]:
# Turn train and test custom Dataset's into DataLoader's
from torch.utils.data import DataLoader
train_dataloader_custom = DataLoader(
    # use custom created train Dataset
    dataset=train_data_custom,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)

test_dataloader_custom = DataLoader(
    dataset=test_data_custom, # use custom created test Dataset
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    # don't usually need to shuffle testing data
    shuffle=False,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)

batch_img, batch_label = next(iter(train_dataloader_custom))
print(batch_img.shape, batch_label.shape)

img, label = batch_img[0], batch_label[0]
print(img.shape, label)

# Model Class (TinyVGG)

To Calculate the required output shape after Flattening:
- https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html > shape
- There's a formula there, but it's also possible to run it with wrong values and correct reading the error

Each Conv2d or MaxPool2D:
- Input: $H_{in}$ (Height), $W_{in}$ (Witdth)
- Output: $H_{out}$ (Height), $W_{out}$ (Witdth)
- dilation: Spacing between kernel elements.
---
Where:
$$
H_{out} = \lfloor \frac{(H_{in} + 2 * padding[0] - dilation[0] * (kernelsize[0] - 1) - 1)}{stride[0]} + 1 \rfloor
$$
and
$$
W_{out} = \lfloor \frac{(W_{in} + 2 * padding[1] - dilation[1] * (kernelsize[1] - 1) - 1)}{stride[1]} + 1 \rfloor
$$

---
These Values affect the image size for every Conv2D or MaxPool in the model
 - After going through all modifying layers, the result will have a certain H x W shape for each sample; therefore the input features for the Linear Layer after the Flattening will be multiplyied be these final H x W

In [ ]:
%%writefile src/tiny_vgg.py

def get_out_shape(
        in_shape: tuple[int, int],
        kernel: tuple[int, int],
        stride: tuple[int, int],
        pad: tuple[int, int],
        dilation: tuple[int, int],
    ) -> tuple[int, int]:

    out = [0, 0]
    for i, inp in enumerate(in_shape):
        a = inp + 2*pad[i]
        b = dilation[i]*(kernel[i] - 1)
        c = (a - b - 1) // stride[i]
        out[i] = 1 + c

    return out[0], out[1]

class TinyVGG(nn.Module):
    """
    Model architecture copying TinyVGG from:
    https://poloclub.github.io/cnn-explainer/
    """
    # For the Conv2D Layers
    KERNEL_SIZE = (3, 3)
    STRIDE = (1, 1)
    PADDING = (1, 1)
    DILATION = (1, 1)
    ## For The MaxPool2D
    KERNEL_SIZE_M = (2, 2)
    STRIDE_M = (2, 2)
    PADDING_M = (0, 0)
    DILATION_M = (1, 1)
    # each layer of the network compresses and changes
    # the shape of the input data.

    def __init__(self,
                 input_shape: int,
                 hidden_units: int,
                 output_shape: int,
                 entry_shape: tuple[int, int] = (64, 64)
        ) -> None:
        super().__init__()

        #### 1º Layer
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=self.KERNEL_SIZE,
                stride=self.STRIDE,
                padding=self.PADDING,
                dilation=self.DILATION,
            ),
            nn.ReLU(),
            nn.Conv2d(
                hidden_units,
                hidden_units,
                self.KERNEL_SIZE,
                self.STRIDE,
                self.PADDING,
                self.DILATION,
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=self.KERNEL_SIZE_M,
                stride=self.STRIDE_M
            )
        )

        # Conv2D 1
        entry_shape = get_out_shape(
            entry_shape,
            self.KERNEL_SIZE,
            self.STRIDE,
            self.PADDING,
            self.DILATION,
        )
        # Conv2D 2
        entry_shape = get_out_shape(
            entry_shape,
            self.KERNEL_SIZE,
            self.STRIDE,
            self.PADDING,
            self.DILATION,
        )
        # MaxPool2D
        entry_shape = get_out_shape(
            entry_shape,
            self.KERNEL_SIZE_M,
            self.STRIDE_M,
            self.PADDING_M,
            self.DILATION_M,
        )

        #### 2º Layer
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, self.KERNEL_SIZE, self.STRIDE, self.PADDING, self.DILATION,),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, self.KERNEL_SIZE, self.STRIDE, self.PADDING,self.DILATION,),
            nn.ReLU(),
            # default stride value is same as kernel_size
            nn.MaxPool2d(self.KERNEL_SIZE_M, self.STRIDE_M)
        )
        # Conv2D 1
        entry_shape = get_out_shape(
            entry_shape,
            self.KERNEL_SIZE,
            self.STRIDE,
            self.PADDING,
            self.DILATION,
        )
        # Conv2D 2
        entry_shape = get_out_shape(
            entry_shape,
            self.KERNEL_SIZE,
            self.STRIDE,
            self.PADDING,
            self.DILATION,
        )
        # MaxPool2D
        entry_shape = get_out_shape(
            entry_shape,
            self.KERNEL_SIZE_M,
            self.STRIDE_M,
            self.PADDING_M,
            self.DILATION_M,
        )
        flattened = entry_shape[0]*entry_shape[0]
        ### Outer Layer
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*flattened,
                      out_features=output_shape)
        )

    def forward(self, x: torch.Tensor):
        # x = self.conv_block_1(x)
        # print(x.shape)
        # x = self.conv_block_2(x)
        # print(x.shape)
        # x = self.classifier(x)
        # print(x.shape)
        # return x
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

# Model 0 (Without Augmentation)



## Loading Data

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

transform_0 = transforms.Compose([
    transforms.Resize(size=(HEIGHT, WIDTH)),
    transforms.ToTensor(),
])

train_dataset_0 = datasets.ImageFolder(
    root=train_dir,
    transform=transform_0,
)
test_dataset_0 = datasets.ImageFolder(
    root=test_dir,
    transform=transform_0,
)
train_data_loader_0 = DataLoader(
    dataset=train_dataset_0,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)
test_data_loader_0 = DataLoader(
    dataset=test_dataset_0,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)

batch_img, batch_label = next(iter(train_data_loader_0))

print("Batches:", batch_img.shape, batch_label.shape)

img, label = batch_img[0], batch_label[0]

COLORS, _, _ = img.shape

print(f"Colors: {COLORS} | Height: {HEIGHT} | Width: {WIDTH}")

## Create Model with the Data

In [ ]:
from src.tiny_vgg import TinyVGG
from src.training import device

torch.manual_seed(RND_SEED)
model_0 = TinyVGG(
    # number of color channels (3 for RGB)
    input_shape=COLORS,
    hidden_units=10,
    output_shape=len(train_data.classes),
    entry_shape=(HEIGHT, WIDTH),
).to(device)

from torchinfo import summary
print(summary(model_0, input_size=batch_img.shape))

In [ ]:
import torch.nn as nn
import torch.optim as optim

from src.helper_functions import accuracy_fn
from src.training import train_model_with_batches, eval_model, plot_loss_curves

model_0_history = train_model_with_batches(
    model=model_0,
    train_data=train_data_loader_0,
    test_data=test_data_loader_0,
    loss_fn=nn.CrossEntropyLoss(),
    optimizer_fn=optim.Adam,
    lr=0.001,
    N_EPOCHS=5,
    RND_SEED=RND_SEED,
    eval_fn=accuracy_fn,
    is_classification=True,
)
model_0_results = eval_model(
    model=model_0,
    data_loader=test_data_loader_0,
    loss_fn=nn.CrossEntropyLoss(),
    device=device,
    eval_fn=accuracy_fn,
)
plot_loss_curves(model_0_history)

# Loss curves

An ideal loss cruve will have both the validation and training dataset progressing close to each other and achieving acceptable results

 - If the training curvew is much better than the testing, it's a signal the model is **overfitting** to the training data, being inable to generalize
 - If both progress at a similar level, but don't achieve good enough results, it's a signal of **underfitting**

## Dealing with Overfitting

1. Get more data samples
2. Data Augmentation
3. Filter poor samples from the dataset
4. Transfer Learning
   - Use another model's weights and tweak them
5. Simplify the model
6. Learning Rate Decay
   - `torch.optim.lr_scheduler`
7. Early Stopping
   - Stop the training on a certain criterion other than the number of epochs

## Dealing with Underfitting

1. Add more layers/units to the model
2. Change the Learning Rate
3. Increase the number of Epochs
4. Transfer Learning
5. Less Regularization
    - Reduce the anti-overfitting methods

# Data Augmentation

Techniques to increase the amount of data by generating new modified copies or syntetich data based on patterns found in existing samples
- It's not **oversampling**, which only copies the data
It is a process of artificially adding diversity to the data

PyTorch has multiple forms of data augmentation, specially for [images](https://docs.pytorch.org/vision/main/auto_examples/transforms/plot_transforms_illustrations.html#augmentation-transforms):
- Usually, random augmenters outperform manual ones, even the [TrivialAugmentWide](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.TrivialAugmentWide.html)
    - WHich simply randomly pick a number of images from a dataset to augment at a random magnitude (between a specified range)
- It has shown very positive [effects](https://pytorch.org/blog/how-to-train-state-of-the-art-models-using-torchvision-latest-primitives/#break-down-of-key-accuracy-improvements)

**_NOTE:_ Never perform data augmentation on the test set!**
- It could biase results

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize(size=(HEIGHT, WIDTH)),
    # the intensity of the transformations
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor(),
])
# don't perform data augmentation on the test set!
# It could biase results
test_transform = transforms.Compose([
    transforms.Resize(size=(HEIGHT, WIDTH)),
    transforms.ToTensor(),
])

# Get all image paths
image_path_list = list(image_path.glob("*/*/*.jpg"))

# Plot random images
plot_transformed_images(
    image_paths=image_path_list,
    transform=train_transform,
    n=3,
)

# Model 1 (With Augmentation)

## Load data

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

transform_1 = transforms.Compose([
    transforms.Resize(size=(HEIGHT, WIDTH)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor(),
])

train_dataset_1 = datasets.ImageFolder(
    root=train_dir,
    transform=transform_1,
)
test_dataset_1 = datasets.ImageFolder(
    root=test_dir,
    transform=transform_0,
)
train_data_loader_1 = DataLoader(
    dataset=train_dataset_1,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)
test_data_loader_1 = DataLoader(
    dataset=test_dataset_1,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False,
    # Writes more data on the GPU,
    # reducing the transfer time
    pin_memory=True,
)

batch_img, batch_label = next(iter(train_data_loader_1))

print("Batches:", batch_img.shape, batch_label.shape)

img, label = batch_img[0], batch_label[0]

print(f"Img: {img.shape} | Label: {label}")

## Creating The Model with Augmented Data

In [ ]:
torch.manual_seed(RND_SEED)
model_1 = TinyVGG(
    # number of color channels (3 for RGB)
    input_shape=COLORS,
    hidden_units=10,
    output_shape=len(train_data.classes),
    entry_shape=(HEIGHT, WIDTH),
).to(device)

from torchinfo import summary
summary(model_1, input_size=batch_img.shape)

In [ ]:
from src.helper_functions import accuracy_fn

model_1_history = train_model_with_batches(
    model=model_1,
    train_data=train_data_loader_1,
    test_data=test_data_loader_1,
    loss_fn=nn.CrossEntropyLoss(),
    optimizer_fn=optim.Adam,
    lr=0.001,
    N_EPOCHS=5,
    RND_SEED=RND_SEED,
    eval_fn=accuracy_fn,
    is_classification=True,
)
model_1_results = eval_model(
    model=model_1,
    data_loader=test_data_loader_1,
    loss_fn=nn.CrossEntropyLoss(),
    device=device,
    eval_fn=accuracy_fn,
)
plot_loss_curves(model_1_history);

# Plotting Comparisons Between Models

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df0 = pd.DataFrame(model_0_history)
df1 = pd.DataFrame(model_1_history)

epochs = range(len(df0))

plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(epochs, df0['train_loss'], label="Model 0")
plt.plot(epochs, df1['train_loss'], label="Model 1")
plt.title("Train Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(epochs, df0['test_loss'], label="Model 0")
plt.plot(epochs, df1['test_loss'], label="Model 1")
plt.title("Test Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(epochs, df0['train_eval'], label="Model 0")
plt.plot(epochs, df1['train_eval'], label="Model 1")
plt.title("Train Eval")
plt.xlabel("Epochs")
plt.ylabel("Acc")
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(epochs, df0['test_eval'], label="Model 0")
plt.plot(epochs, df1['test_eval'], label="Model 1")
plt.title("Test Eval")
plt.xlabel("Epochs")
plt.ylabel("Acc")
plt.legend()


## Custom Predictions on the Models

In [ ]:
# Download custom image
import requests

# Setup custom image path
custom_image_path = data_path / "04-pizza-dad.jpeg"

# Download the image if it doesn't already exist
if not custom_image_path.is_file():
    with open(custom_image_path, "wb") as f:
        # When downloading from GitHub, need to use the "raw" file link
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
        print(f"Downloading {custom_image_path}...")
        f.write(request.content)
else:
    print(f"{custom_image_path} already exists, skipping download.")

In [ ]:
import torchvision
from torchvision.io import read_image

# Read in custom image
custom_image = torchvision.io.read_image(str(custom_image_path))

print(f"Custom image dtype: {custom_image.dtype}")
# Converts it to torch dtype
custom_image = custom_image.type(torch.float)
print(f"Custom image dtype: {custom_image.dtype}")

# Scale it from [0, 255] to [0, 1]
# Matplot lib also expect them this way
custom_image = custom_image / 255

# Print out image data
print(f"Custom image shape: {custom_image.shape}\n")

from torchvision.transforms import Compose, Resize, ToTensor

transformer_custom = Compose([
    Resize(size=(HEIGHT, WIDTH)),
])

custom_image = transformer_custom(custom_image)
# Print out image data
print(f"Custom image shape: {custom_image.shape}\n")

plt.imshow(custom_image.permute(1, 2, 0))
plt.axis(False)

# Ensuring it has a Batch Size
custom_image = custom_image.unsqueeze(0)

#Putting it on the same device
custom_image = custom_image.to(device)

model_0.eval()
model_1.eval()
with torch.inference_mode():
    pred0 = model_0(custom_image)
    print(class_names[pred0.softmax(dim=1).argmax(dim=1)])
    pred1 = model_1(custom_image)
    print(class_names[pred1.softmax(dim=1).argmax(dim=1)])